In [13]:
# Block 1: Imports and Directory Setup

import pandas as pd
from pathlib import Path
import os

raw_data_directory = "raw_data" # Verilerin olduğu klasörün ismi


output_directory = "processed_data"  # İşlenmiş verilerin kaydedileceği yer

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

print("Setup completed.")

Setup completed.


In [14]:
# Block 2: File Discovery Check

# Bulunan csv dosyalarını listeleme.
raw_path = Path(raw_data_directory)
files = list(raw_path.glob("*.csv"))

print(f"Found {len(files)} CSV files in '{raw_data_directory}'")
for f in files:
    print("-", f.name)

if len(files) == 0:
    print("!!! HATA: Dosyalar bulunamadı. Lütfen Block 1'deki klasör yolunu kontrol et.")

Found 20 CSV files in 'raw_data'
- P01_gender_task_2026-01-04_01h41.41.014.csv
- P02_gender_task_2025-12-24_17h09.50.142.csv
- P03_gender_task_2026-01-01_21h50.27.704.csv
- P04_gender_task_2025-12-18_21h51.24.625.csv
- P05_gender_task_2025-12-20_03h07.34.945.csv
- P06_gender_task_2026-01-04_01h50.35.866.csv
- P07_gender_task_2026-01-06_17h59.02.517.csv
- P08_gender_task_2025-12-25_16h15.28.233.csv
- P16_gender_task_2026-01-04_15h36.11.905.csv
- P09_gender_task_2026-01-03_22h46.50.968.csv
- P10_gender_task_2025-12-18_22h27.06.924.csv
- P11_gender_task_2025-12-30_16h58.22.306.csv
- P12_gender_task_2025-12-20_02h49.18.355.csv
- P17_gender_task_2026-01-06_21h30.44.056.csv
- P13_gender_task_2025-12-18_22h36.47.025.csv
- P14_gender_task_2026-01-06_11h32.00.456.csv
- P18_gender_task_2025-12-18_13h15.20.038.csv
- P15_gender_task_2025-12-24_17h15.12.355.csv
- P19_gender_task_2025-12-24_17h20.02.367.csv
- P20_gender_task_2025-12-20_02h58.43.920.csv


In [15]:
# Block 3: Loading and Filtering Participants
all_data_list = []

for file_path in files:
    # PsychoPy çıktıları bazen 'utf-8-sig' kodlaması gerektirir
    temp_df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 1. Filtre: 'P18' dosyasını atla, bu dosya deneyi deneme amaçlı yapıldığı için analiz kısmına dahil edilmemiştir.
    if "P18" in file_path.name:
        continue
    
    # 2. Filtre: Trial olmayan satırları temizle (ImageFile sütunu boş olanlar)
    temp_df = temp_df.dropna(subset=['ImageFile'])
    
    # 3. Filtre: Yaş kriterini kontrol et (18-60) + yaş bilgisi okunamazsa katılımcıyı dışla
    # (Katılımcı bazlı yaş verisi genellikle her satırda aynıdır, ilk satırdan bakıyoruz)

    # Katılımcı adını güvenli şekilde al (olmazsa dosya adını kullan)
    participant_name = (
        temp_df['participant'].iloc[0]
        if 'participant' in temp_df.columns and not temp_df['participant'].isna().all()
        else file_path.stem
    )

    # Yaş bilgisini güvenli şekilde sayıya çevir (okunamazsa NaN olur)
    age_val = (
        pd.to_numeric(temp_df['age'].iloc[0], errors='coerce')
        if 'age' in temp_df.columns
        else pd.NA
    )

    # Yaş okunamaz / eksikse: bu katılımcıyı analizden çıkar
    if pd.isna(age_val):
        print(f"Excluding subject {participant_name} (Age: missing/unreadable) | File: {file_path.name}")
        continue

    participant_age = int(age_val)

    # Yaş aralığı kriteri (18-60)
    if participant_age < 18:
        print(f"Excluding subject {participant_name} (Age: {participant_age})")
        continue

    if participant_age > 60:
        print(f"Excluding subject {participant_name} (Age: {participant_age})")
        continue


    # Sadece analizde kullanacağımız sütunları seçiyoruz.
    columns_to_keep = [
    'participant', 
    'age', 
    'handedness',      
    'gender',        
    'Gender',          # Yüzün cinsiyeti (Uyaran değişkeni,isimlendirmedeki bu karışıklık ilerde düzeltilecektir)
    'Emotion',         
    'key_resp.rt', 
    'key_resp.corr'
    ]
    
    all_data_list.append(temp_df[columns_to_keep])

print(f"Successfully processed {len(all_data_list)} participants.")

Excluding subject P04 (Age: 14)
Successfully processed 18 participants.


**NOT: "Gender" ve "gender" karmaşıklığı anova dosyasında yeniden adlandırma ile düzeltilmiştir.**

In [16]:
# Block 4: Merging and Exporting Combined Data
if len(all_data_list) > 0:
    combined_df = pd.concat(all_data_list, ignore_index=True)
    
    # Kayıt işlemi
    save_path = os.path.join(output_directory, "combined_raw_data.csv")
    combined_df.to_csv(save_path, index=False)
    
    print(f"Merged data saved to: {save_path}")
    display(combined_df.head()) # Verinin ilk 5 satırını gör
else:
    print("No data to merge. Please check the exclusion criteria in Block 3.")

Merged data saved to: processed_data\combined_raw_data.csv


,participant,age,handedness,gender,Gender,Emotion,key_resp.rt,key_resp.corr
0,P01,23,Right,Male,Female,Neutral,0.997217,1.0
1,P01,23,Right,Male,Female,Happy,1.587789,0.0
2,P01,23,Right,Male,Female,Happy,1.110738,1.0
3,P01,23,Right,Male,Male,Angry,0.747143,1.0
4,P01,23,Right,Male,Female,Happy,0.468701,1.0
